# De-mystifying GPT Attention & The Causal Mask

This notebook breaks down the `CausalSelfAttention` mechanism. We will use small, dummy data to visualize exactly how the "Triangle Trick" works.

### 1\. Setup and Dummy Data

Let's create a fake scenario.

  * **Batch (B) = 1**: Just one sequence for now.
  * **Time (T) = 8**: A sentence with 8 tokens (e.g., "The cat sat on the mat and...").
  * **Channels (C) = 32**: The embedding size.
  * **Head Size = 16**: We will pretend we are looking at just **one head**.

In [1]:
import torch
import torch.nn.functional as F
import math
import matplotlib.pyplot as plt # Optional, for visualizing if you want

torch.manual_seed(1337) # For reproducibility

B, T, C = 1, 8, 32 # Batch, Time, Channels
head_size = 16

# Create random Query and Key vectors for our 8 tokens
# In a real model, these come from the Linear layers (qkv_proj)
q = torch.randn(B, T, head_size)  # What tokens are LOOKING for
k = torch.randn(B, T, head_size)  # What tokens CONTAIN
v = torch.randn(B, T, head_size)  # The information to pass along

### 2\. The Raw Affinity (Similarity Scores)

First, every token looks at every other token. We do this via a **Dot Product**.

  * If a Query and a Key are similar, the dot product is **high** (positive).
  * If they are unrelated, the dot product is **near zero**.
  * If they are opposites, the dot product is **negative**.

In [2]:
# We transpose K so we can dot product (B, T, hs) @ (B, hs, T) -> (B, T, T)
wei = q @ k.transpose(-2, -1) 

# Scale by sqrt(head_size) to keep variance stable (explained in previous answers)
wei = wei * (head_size ** -0.5)

print("Raw Attention Scores (The 8x8 Matrix):")
# We only look at the first batch
print(wei[0].round(decimals=2))

Raw Attention Scores (The 8x8 Matrix):
tensor([[-0.1600, -0.9400, -0.7000,  0.2200, -1.2800,  0.9800, -0.4200,  1.6700],
        [-0.9100,  0.2600, -1.0800, -0.7500,  0.2100, -0.4700, -0.1700,  1.0600],
        [ 0.6500, -0.3800,  0.9900,  1.2900,  1.0100,  0.0500,  1.8400, -0.8600],
        [ 0.4100, -0.7100, -1.4000, -0.5200, -0.4600,  0.4900,  1.2600, -0.6500],
        [ 1.1300, -0.8400,  0.6800,  0.6400,  0.1500, -0.2000,  0.9600, -1.3900],
        [-2.8300,  0.6300,  1.1300, -0.8500,  0.9500,  0.2700,  0.3200,  0.7400],
        [-0.9400, -1.4900,  1.6100,  0.7200,  0.0800,  0.0000, -0.2100, -2.9200],
        [ 0.1700, -0.1300,  0.3800,  1.5900, -0.0100,  0.5000,  1.2100, -0.5600]])


What you see above: A generic $8 \times 8$ grid of numbers.

- Row 0 is the 1st token. It has a score for every column (0 to 7).
- The Problem: Look at Row 0 (the first token). It has a score for Column 7 (the last token).
- Translation: The first word is "looking at" the future! In a text generation task, this is cheating. You can't know the end of the sentence when you are writing the start.

### 3\. The "Triangle Trick" (Masking)

This is the heart of GPT. We must force the model to ignore the future.
We create a **mask** that has `1`s for the past/present and `0`s for the future.

In [3]:
# Create a lower-triangular matrix of ones
tril = torch.tril(torch.ones(T, T))

print("\nThe Triangle Mask:")
print(tril)


The Triangle Mask:
tensor([[1., 0., 0., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1.]])


Output Explanation: Notice the shape.

- Row 0 has one 1 (Can only see index 0).
- Row 1 has two 1s (Can see index 0 and 1).
- Row 7 has eight 1s (Can see everyone).

### 4\. Applying the Mask

Now we combine our raw scores with the mask.
**The Logic:**
Where the mask is `0` (Future), we replace the score with `-inf` (Negative Infinity).

In [4]:
# masked_fill(condition, value_if_true)
# "Where the mask is 0, make the weight -infinity"
wei = wei.masked_fill(tril == 0, float('-inf'))

print("\nScores after Masking (Notice the -inf):")
print(wei[0].round(decimals=2))


Scores after Masking (Notice the -inf):
tensor([[-0.1600,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [-0.9100,  0.2600,    -inf,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.6500, -0.3800,  0.9900,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.4100, -0.7100, -1.4000, -0.5200,    -inf,    -inf,    -inf,    -inf],
        [ 1.1300, -0.8400,  0.6800,  0.6400,  0.1500,    -inf,    -inf,    -inf],
        [-2.8300,  0.6300,  1.1300, -0.8500,  0.9500,  0.2700,    -inf,    -inf],
        [-0.9400, -1.4900,  1.6100,  0.7200,  0.0800,  0.0000, -0.2100,    -inf],
        [ 0.1700, -0.1300,  0.3800,  1.5900, -0.0100,  0.5000,  1.2100, -0.5600]])


Look at the output! The upper-right triangle is now entirely -inf. The future has been deleted.

### 5\. The Softmax (Normalization)

Why did we use `-inf`? Because of how **Softmax** works.
$$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$$

  * $e^{0} = 1$
  * $e^{\text{high number}} = \text{Huge}$
  * $e^{-\infty} = 0$

When we exponentiate negative infinity, it becomes exactly **zero**.

In [5]:
wei = F.softmax(wei, dim=-1)

print("\nFinal Probabilities (Softmax):")
print(wei[0].round(decimals=2))


Final Probabilities (Softmax):
tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2400, 0.7600, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3600, 0.1300, 0.5100, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5300, 0.1700, 0.0900, 0.2100, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3600, 0.0500, 0.2300, 0.2200, 0.1400, 0.0000, 0.0000, 0.0000],
        [0.0100, 0.2000, 0.3300, 0.0500, 0.2800, 0.1400, 0.0000, 0.0000],
        [0.0400, 0.0200, 0.4700, 0.2000, 0.1000, 0.0900, 0.0800, 0.0000],
        [0.0800, 0.0600, 0.1000, 0.3300, 0.0700, 0.1100, 0.2200, 0.0400]])


Interpretation of the Output: 

Look at Row 0 again.

- Column 0 is 1.00 (100%).
- Columns 1-7 are 0.00. The first token attends only to itself.

Look at Row 1.

- It has probabilities split between Column 0 and Column 1.
- Columns 2-7 are 0.00.

This is the triangular flow of information. Information flows from the past to the present, but never backwards from the future.

### 6\. Aggregating the Values

Finally, we calculate the result. We multiply these probabilities by the `Value` vector ($V$).

  * If I have 0.7 attention on Token A and 0.3 on Token B, my output is a mix: $0.7 \times A + 0.3 \times B$.

In [6]:
out = wei @ v

print("\nFinal Output Shape:")
print(out.shape) # Should be (B, T, Head_Size)


Final Output Shape:
torch.Size([1, 8, 16])


### Summary Diagram

If you were to visualize the matrix `wei` at the end, it looks like this (white is data, black is zero):

```
Row 0: [1, 0, 0, 0]  <- Token 0 only sees itself
Row 1: [0.5, 0.5, 0, 0]  <- Token 1 sees 0 and 1
Row 2: [0.2, 0.2, 0.6, 0]  <- Token 2 sees 0, 1, 2
Row 3: [0.1, 0.1, 0.1, 0.7]  <- Token 3 sees 0, 1, 2, 3
```

This ensures that when we train the model, we can feed it the whole sentence at once (Parallel Training), but mathematically, every position acts like it hasn't seen the future yet.